# DAVI CA2 — Data cleaning and validation

This notebook creates the analysis-ready dataset for the IT systems integrator analysis. The intended source grain is one **Client ID–Year** record. Survey and Revenue/COGS records are joined on both `CLIENT ID` and `YEAR`; joining on `CLIENT ID` alone would create invalid many-to-many combinations and duplicate financial values.

In [ ]:
# Import libraries and define the input/output paths.
import numpy as np
import pandas as pd
from pathlib import Path

pd.set_option('display.max_columns', None)
DATA_DIR = Path('Datasets')
OUTPUT_PATH = Path('merged_cleaned.xlsx')

## 1. Load the source datasets

The three source files are kept separate until their keys and duplicate records have been checked.

In [ ]:
# Load the three source tables and record their starting row counts.
profile = pd.read_excel(DATA_DIR / 'Client Profiles.xlsx')
survey = pd.read_excel(DATA_DIR / 'Client Survey.xlsx')
cogs = pd.read_excel(DATA_DIR / 'Revenue and COGS.xlsx')
original_counts = {
    'Client Profiles': len(profile),
    'Client Survey': len(survey),
    'Revenue and COGS': len(cogs),
}
print('Original row counts:', original_counts)

## 2. Standardise profile fields

Country is derived from the Singapore-based indicator, and staff-strength labels are standardised for consistent grouping.

In [ ]:
# Standardise profile categories used for consistent grouping.
profile['SG-BASED'] = profile['SG-BASED'].astype('string').str.strip()
profile['COUNTRY'] = np.where(
    profile['SG-BASED'].str.lower().isin(['yes', 'y']),
    'Singapore',
    profile['COUNTRY OF OVERSEAS CLIENT'],
)
profile['COUNTRY'] = profile['COUNTRY'].replace({'Malaysian': 'Malaysia'})
profile['STAFF STRENGTH'] = profile['STAFF STRENGTH'].replace({
    'More than 200': '> 200',
    'Less than 200': '50 ~ 200',
    'Less than 49': '1 ~ 49',
})

## 3. Remove exact duplicate source rows and validate keys

Exact duplicate rows are removed from the survey and financial sources. The analytical key is then checked: each source must contain at most one record per `CLIENT ID`–`YEAR`.

In [ ]:
# Remove exact duplicates and verify the client-year key is unique.
survey_exact_duplicates = int(survey.duplicated().sum())
cogs_exact_duplicates = int(cogs.duplicated().sum())
survey = survey.drop_duplicates().reset_index(drop=True)
cogs = cogs.drop_duplicates().reset_index(drop=True)
print('Exact duplicate rows removed:', {
    'Client Survey': survey_exact_duplicates,
    'Revenue and COGS': cogs_exact_duplicates,
})

for name, frame in [('survey', survey), ('cogs', cogs)]:
    duplicate_keys = int(frame.duplicated(['CLIENT ID', 'YEAR']).sum())
    print(f'{name} duplicate Client ID–Year rows:', duplicate_keys)
    if duplicate_keys:
        raise ValueError(f'{name} still contains duplicate Client ID–Year rows')

## 4. Join at the correct Client ID–Year grain

Survey and financial records are matched using both `CLIENT ID` and `YEAR`. This prevents a survey record from being paired with financial records from unrelated years. The `validate` arguments make an invalid merge fail rather than silently producing a many-to-many result.

In [ ]:
# Validate schemas, remove invalid IDs, and join survey data to financial data.
required_profile = {'CLIENT ID', 'TYPE', 'COMMENCEMENT DATE', 'STAFF STRENGTH', 'SECTOR', 'COUNTRY'}
required_survey = {'CLIENT ID', 'YEAR', 'PRESALES AND PARTNERSHIP', 'TECHNICAL EXPERTISE', 'PROJECT DELIVERY', 'POST-SALES SUPPORT', 'NPS RATING'}
required_cogs = {'CLIENT ID', 'YEAR', 'REVENUE', 'HARDWARE', 'SOFTWARE', 'MANPOWER'}
for name, frame, required in [('profile', profile, required_profile), ('survey', survey, required_survey), ('cogs', cogs, required_cogs)]:
    missing = sorted(required - set(frame.columns))
    if missing:
        raise KeyError(f'{name} is missing required columns: {missing}')

profile_client_ids = set(profile['CLIENT ID'])
invalid_survey_ids = int((~survey['CLIENT ID'].isin(profile_client_ids)).sum())
invalid_cogs_ids = int((~cogs['CLIENT ID'].isin(profile_client_ids)).sum())
survey = survey[survey['CLIENT ID'].isin(profile_client_ids)].copy()
cogs = cogs[cogs['CLIENT ID'].isin(profile_client_ids)].copy()

annual = survey.merge(
    cogs,
    on=['CLIENT ID', 'YEAR'],
    how='inner',
    validate='one_to_one',
    indicator=True,
)
print('Invalid survey Client IDs removed:', invalid_survey_ids)
print('Invalid COGS Client IDs removed:', invalid_cogs_ids)
print('Matched Client ID–Year records:', len(annual))

## 5. Create derived fields and final analysis table

Gross profit is Revenue minus Hardware, Software and Manpower. NPS categories follow the standard thresholds used in the project: Promoter 9–10, Passive 7–8 and Detractor 0–6.

In [ ]:
# Join profile attributes, coerce types, and derive financial and NPS fields.
profile_columns = ['CLIENT ID', 'TYPE', 'COMMENCEMENT DATE', 'STAFF STRENGTH', 'SECTOR', 'COUNTRY']
df = profile[profile_columns].merge(
    annual.drop(columns=['_merge']),
    on='CLIENT ID',
    how='inner',
    validate='one_to_many',
)

df['COMMENCEMENT DATE'] = pd.to_datetime(df['COMMENCEMENT DATE'], errors='coerce')
numeric_columns = ['YEAR', 'PRESALES AND PARTNERSHIP', 'TECHNICAL EXPERTISE', 'PROJECT DELIVERY', 'POST-SALES SUPPORT', 'NPS RATING', 'REVENUE', 'HARDWARE', 'SOFTWARE', 'MANPOWER']
for column in numeric_columns:
    df[column] = pd.to_numeric(df[column], errors='coerce')
df['COGS'] = df[['HARDWARE', 'SOFTWARE', 'MANPOWER']].sum(axis=1)
df['GROSS PROFIT'] = df['REVENUE'] - df['COGS']
df['GROSS MARGIN'] = df['GROSS PROFIT'] / df['REVENUE'].replace(0, np.nan)
df['NPS CATEGORY'] = np.select(
    [df['NPS RATING'].ge(9), df['NPS RATING'].ge(7)],
    ['Promoter', 'Passive'],
    default='Detractor',
)
df = df.sort_values(['CLIENT ID', 'YEAR']).reset_index(drop=True)
print(f'Final dataset: {len(df):,} rows, {df["CLIENT ID"].nunique():,} unique clients, years {df["YEAR"].min()}–{df["YEAR"].max()}')

## 6. Data-quality summary and export

The final checks confirm the declared grain, missing-value status and financial reconciliation before exporting the cleaned workbook.

In [ ]:
# Summarise quality checks, assert the final table is valid, and export it.
quality_summary = pd.DataFrame({
    'Metric': [
        'Original profile records', 'Original survey records', 'Original COGS records',
        'Exact survey duplicates removed', 'Exact COGS duplicates removed',
        'Invalid survey Client IDs removed', 'Invalid COGS Client IDs removed',
        'Final Client ID–Year records', 'Final unique clients',
        'Duplicate final Client ID–Year records', 'Missing final cells',
    ],
    'Value': [
        original_counts['Client Profiles'], original_counts['Client Survey'], original_counts['Revenue and COGS'],
        survey_exact_duplicates, cogs_exact_duplicates, invalid_survey_ids, invalid_cogs_ids,
        len(df), df['CLIENT ID'].nunique(), int(df.duplicated(['CLIENT ID', 'YEAR']).sum()), int(df.isna().sum().sum()),
    ],
})
display(quality_summary)

if df.duplicated(['CLIENT ID', 'YEAR']).any():
    raise ValueError('Final dataset contains duplicate Client ID–Year records.')
if df.isna().any().any():
    raise ValueError('Final dataset contains missing values.')
if not np.isclose(df['GROSS PROFIT'].sum(), (df['REVENUE'] - df['HARDWARE'] - df['SOFTWARE'] - df['MANPOWER']).sum()):
    raise ValueError('Gross-profit reconciliation failed.')

df.to_excel(OUTPUT_PATH, index=False)
print(f'Exported cleaned dataset to {OUTPUT_PATH.resolve()}')